In [ ]:
# ===== E-Commerce Customer Segmentation & Recommendation (single cell) =====
import os, glob, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

sns.set_style("whitegrid")
K = 4   # number of segments (change after looking at the elbow/silhouette plots, then re-run)

# ---------- 1. Load data ----------
URL = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
FILE = "Online Retail.xlsx"
if not os.path.exists(FILE):
    try:
        req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=60) as r, open("online_retail.zip", "wb") as f:
            f.write(r.read())
        with zipfile.ZipFile("online_retail.zip") as z:
            z.extractall(".")
        print("Downloaded:", FILE)
    except Exception as e:
        print("Auto-download failed:", e, "\nUpload Online Retail.xlsx (or data.csv):")
        from google.colab import files
        FILE = list(files.upload().keys())[0]

raw = pd.read_excel(FILE) if FILE.endswith(".xlsx") else pd.read_csv(FILE, encoding="ISO-8859-1")
raw.columns = [c.strip() for c in raw.columns]
print("Raw shape:", raw.shape)

# ---------- 2. Clean transactions ----------
df = raw.copy()
n0 = len(df)
df = df.dropna(subset=["CustomerID"])
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
NON_PRODUCT = ["POST", "D", "M", "DOT", "CRUK", "BANK CHARGES", "AMAZONFEE", "S", "B", "PADS"]
df = df[~df["StockCode"].astype(str).isin(NON_PRODUCT)]
df = df.drop_duplicates()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["CustomerID"] = df["CustomerID"].astype(int)
df["StockCode"] = df["StockCode"].astype(str)
df["Description"] = df["Description"].astype(str).str.strip()
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]
df = df.reset_index(drop=True)
print(f"Cleaned rows: {n0} -> {len(df)} | Customers: {df['CustomerID'].nunique()} | Products: {df['StockCode'].nunique()}")
df.to_csv("cleaned_transactions.csv", index=False)

# ---------- 3. Customer-level features (RFM) ----------
snapshot = df["InvoiceDate"].max() + pd.Timedelta(days=1)
customers = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalPrice", "sum"),
    UniqueProducts=("StockCode", "nunique"),
    TotalItems=("Quantity", "sum"),
).reset_index()
customers["AvgOrderValue"] = customers["Monetary"] / customers["Frequency"]
customers.to_csv("customer_features.csv", index=False)

# ---------- 4. Scaling ----------
rfm_cols = ["Recency", "Frequency", "Monetary"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for i, col in enumerate(rfm_cols):
    sns.histplot(customers[col], bins=40, ax=axes[0, i], color="steelblue"); axes[0, i].set_title(f"{col} (original)")
    sns.histplot(np.log1p(customers[col]), bins=40, ax=axes[1, i], color="seagreen"); axes[1, i].set_title(f"log1p({col})")
plt.tight_layout(); plt.savefig("rfm_distributions.png", dpi=150, bbox_inches="tight"); plt.show()
X = StandardScaler().fit_transform(np.log1p(customers[rfm_cols]))

# ---------- 5. Elbow + silhouette ----------
ks = list(range(2, 9)); inertias, sils = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(km.inertia_); sils.append(silhouette_score(X, km.labels_))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ks, inertias, marker="o"); axes[0].set_title("Elbow Method"); axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia")
axes[1].plot(ks, sils, marker="o", color="darkorange"); axes[1].set_title("Silhouette Score"); axes[1].set_xlabel("k"); axes[1].set_ylabel("Score")
plt.tight_layout(); plt.savefig("elbow_silhouette.png", dpi=150, bbox_inches="tight"); plt.show()
for k, s in zip(ks, sils):
    print(f"k={k}: silhouette={s:.3f}")

# ---------- 6. K-Means segmentation ----------
customers["Cluster"] = KMeans(n_clusters=K, n_init=10, random_state=42).fit_predict(X)
print(f"\nChosen K = {K} | silhouette = {silhouette_score(X, customers['Cluster']):.3f}")

# ---------- 7. Segment profiles + business names ----------
profile = customers.groupby("Cluster").agg(
    Customers=("CustomerID", "count"), Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"), Avg_Monetary=("Monetary", "mean"),
    Total_Revenue=("Monetary", "sum"))
score = (-profile["Avg_Recency"]).rank() + profile["Avg_Frequency"].rank() + profile["Avg_Monetary"].rank()
ordered = score.sort_values(ascending=False).index.tolist()
NAMES = ["Champions", "Regular Customers", "Occasional Buyers", "At-Risk Customers"]
seg_name = {cl: (NAMES[i] if i < len(NAMES) else f"Segment {i+1}") for i, cl in enumerate(ordered)}
customers["Segment"] = customers["Cluster"].map(seg_name)
profile["Segment"] = profile.index.map(seg_name)
profile["Customer_Share_%"] = profile["Customers"] / profile["Customers"].sum() * 100
profile["Revenue_Share_%"] = profile["Total_Revenue"] / profile["Total_Revenue"].sum() * 100
profile = profile.set_index("Segment").loc[[seg_name[c] for c in ordered]].round(1)
print("\n", profile.to_string())
profile.to_csv("segment_profiles.csv"); customers.to_csv("customer_segments.csv", index=False)

# ---------- 8. 2D visualization ----------
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X)
customers["PC1"], customers["PC2"] = coords[:, 0], coords[:, 1]
print("\nVariance explained by 2 PCs:", pca.explained_variance_ratio_.sum().round(3))
order = [seg_name[c] for c in ordered]
plt.figure(figsize=(9, 6))
sns.scatterplot(data=customers, x="PC1", y="PC2", hue="Segment", hue_order=order, palette="Set2", s=25, alpha=0.8)
plt.title("Customer Segments (PCA of scaled RFM)"); plt.savefig("customer_segments_2d.png", dpi=150, bbox_inches="tight"); plt.show()
plt.figure(figsize=(9, 6))
sns.scatterplot(data=customers, x="Recency", y="Monetary", hue="Segment", hue_order=order, palette="Set2", s=25, alpha=0.8)
plt.yscale("log"); plt.title("Recency vs Monetary by Segment")
plt.savefig("recency_vs_monetary.png", dpi=150, bbox_inches="tight"); plt.show()

# ---------- 9. Recommendations (cosine similarity) ----------
cust_cat = customers["CustomerID"].astype("category")
prod_cat = df["StockCode"].astype("category")
pairs = df[["CustomerID", "StockCode"]].drop_duplicates()
rows = pd.Categorical(pairs["CustomerID"], categories=cust_cat.cat.categories).codes
cols = pd.Categorical(pairs["StockCode"], categories=prod_cat.cat.categories).codes
matrix = csr_matrix((np.ones(len(pairs)), (rows, cols)),
                    shape=(len(cust_cat.cat.categories), len(prod_cat.cat.categories)))
cust_index = {c: i for i, c in enumerate(cust_cat.cat.categories)}
prod_list = list(prod_cat.cat.categories)
prod_index = {p: i for i, p in enumerate(prod_list)}
desc_map = df.groupby("StockCode")["Description"].agg(lambda s: s.value_counts().index[0]).to_dict()
item_sim = cosine_similarity(matrix.T)

def recommend_similar_products(stock_code, n=5):
    """Item-based: products most similar to the given product."""
    stock_code = str(stock_code)
    if stock_code not in prod_index:
        raise ValueError(f"StockCode {stock_code} not found")
    i = prod_index[stock_code]
    sims = item_sim[i].copy(); sims[i] = -1
    top = np.argsort(sims)[::-1][:n]
    return pd.DataFrame({"StockCode": [prod_list[j] for j in top],
                         "Description": [desc_map[prod_list[j]] for j in top],
                         "Similarity": sims[top].round(3)})

def recommend_for_customer(customer_id, n=5, n_neighbors=20):
    """Customer-based: what similar customers bought that this customer has not."""
    if customer_id not in cust_index:
        raise ValueError(f"CustomerID {customer_id} not found")
    i = cust_index[customer_id]
    sims = cosine_similarity(matrix[i], matrix).ravel(); sims[i] = 0
    nb = np.argsort(sims)[::-1][:n_neighbors]
    scores = np.asarray(matrix[nb].T.dot(sims[nb])).ravel()
    scores[matrix[i].toarray().ravel() > 0] = 0
    top = np.argsort(scores)[::-1][:n]
    return pd.DataFrame({"StockCode": [prod_list[j] for j in top],
                         "Description": [desc_map[prod_list[j]] for j in top],
                         "Score": scores[top].round(3)})

demo_product = df["StockCode"].value_counts().index[0]
demo_customer = int(customers.sort_values("Monetary", ascending=False).iloc[0]["CustomerID"])
print(f"\nProducts similar to {demo_product} ({desc_map[demo_product]}):")
print(recommend_similar_products(demo_product).to_string(), "\n")
print(f"Recommendations for customer {demo_customer}:")
print(recommend_for_customer(demo_customer).to_string())

# ---------- 10. Business recommendations per segment ----------
ACTIONS = {
    "Champions": "Reward them: loyalty perks, early access, referral offers. Avoid over-discounting.",
    "Regular Customers": "Grow them: bundles and cross-sell suggestions to move them towards Champions.",
    "Occasional Buyers": "Activate them: seasonal campaigns, free-shipping thresholds, purchase reminders.",
    "At-Risk Customers": "Win them back: personalised re-engagement emails and comeback discounts.",
}
for seg, row in profile.iterrows():
    print(f"\n{seg}: {int(row['Customers'])} customers "
          f"({row['Customer_Share_%']}% of customers, {row['Revenue_Share_%']}% of revenue)")
    print(f"  Avg recency {row['Avg_Recency']} days | frequency {row['Avg_Frequency']} orders | spend {row['Avg_Monetary']}")
    print("  Action:", ACTIONS.get(seg, "Review this segment manually."))

# ---------- 11. Zip + download all outputs ----------
out_files = [f for f in glob.glob("*.csv") + glob.glob("*.png")]
with zipfile.ZipFile("project_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in out_files:
        z.write(f)
try:
    from google.colab import files
    files.download("project_outputs.zip")
except ImportError:
    print("Zipped:", out_files)